# LLM Evaluation — Without Description

Predict loan outcomes using an LLM with **structured features only** (no borrower description).
Compare results against the XGBoost model on the same 100 test samples.

## Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from llm_utils import (
    load_llm_sample, run_ml_on_sample,
    build_system_prompt, build_user_prompt,
    call_llm, parse_llm_response,
    evaluate_predictions, compare_results,
    RESULTS_DIR
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# API key loaded automatically from .env file
API_PROVIDER = "gemini"            # "gemini", "anthropic", or "openai"
MODEL_NAME   = "gemini-2.0-flash"  # None = use default for provider
API_KEY      = None                # None = read from .env / environment

## Load Data & Run XGBoost

In [ ]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

print(f"Sample size: {len(llm_sample)}")
print(f"Class distribution:\n{llm_sample['loan_status'].value_counts()}")

In [ ]:
xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)
print(f"XGBoost predictions ready: {len(xgb_preds)} samples")

## LLM Predictions (No Description)

In [ ]:
system_prompt = build_system_prompt()

# Preview the prompt for the first loan
sample_prompt = build_user_prompt(llm_sample.iloc[0], include_desc=False)
print("System prompt:")
print(system_prompt)
print("\n" + "=" * 50)
print("\nSample user prompt:")
print(sample_prompt)

In [ ]:
# Run LLM on all 100 samples
llm_predictions = []
llm_reasonings = []
llm_raw_responses = []

for i, (_, row) in enumerate(tqdm(llm_sample.iterrows(), total=len(llm_sample))):
    user_prompt = build_user_prompt(row, include_desc=False)

    raw = call_llm(
        system_prompt, user_prompt,
        api_provider=API_PROVIDER, model=MODEL_NAME, api_key=API_KEY
    )
    llm_raw_responses.append(raw)

    parsed = parse_llm_response(raw)
    llm_predictions.append(parsed['prediction'])
    llm_reasonings.append(parsed['reasoning'])

print(f"\nCompleted: {len(llm_predictions)} predictions")
print(f"Parse errors: {sum(1 for p in llm_predictions if p is None)}")

## Evaluation

In [ ]:
llm_metrics = evaluate_predictions(y_true, llm_predictions, label="LLM (No Desc)")
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(), label="XGBoost")

In [ ]:
comparison = compare_results(y_true, llm_predictions, xgb_preds.tolist(), llm_reasonings)

print(f"LLM accuracy: {comparison['llm_correct'].mean()*100:.1f}%")
print(f"XGBoost accuracy: {comparison['xgb_correct'].mean()*100:.1f}%")
print(f"\nAgreement between LLM and XGBoost: {(comparison['llm_pred'] == comparison['xgb_pred']).mean()*100:.1f}%")

comparison.head(10)

In [ ]:
# Cases where LLM and XGBoost disagree
disagree = comparison[comparison['llm_pred'] != comparison['xgb_pred']]
print(f"Disagreements: {len(disagree)} / {len(comparison)}")
print(f"LLM correct in disagreements: {disagree['llm_correct'].sum()}")
print(f"XGBoost correct in disagreements: {disagree['xgb_correct'].sum()}")

if len(disagree) > 0:
    print("\nSample disagreements with LLM reasoning:")
    for _, row in disagree.head(5).iterrows():
        actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
        llm = 'Fully Paid' if row['llm_pred'] == 1 else 'Charged Off'
        xgb = 'Fully Paid' if row['xgb_pred'] == 1 else 'Charged Off'
        print(f"  Actual: {actual} | LLM: {llm} | XGBoost: {xgb}")
        print(f"  Reasoning: {row['llm_reasoning']}\n")

## Export Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

comparison.to_csv(f"{RESULTS_DIR}/04a_llm_no_desc_results.csv", index=False)

summary = pd.DataFrame([llm_metrics, xgb_metrics],
                        index=['LLM (No Desc)', 'XGBoost'])
summary.to_csv(f"{RESULTS_DIR}/04a_llm_no_desc_metrics.csv")
print(summary.to_string())